### Data Ingestion

In [2]:
from langchain_core.documents import Document

In [6]:
doc = Document(
    page_content = "This is a random document",
    metadata = {
        "source":"randomtxt.txt",
        "page":1
    }
)
doc

Document(metadata={'source': 'randomtxt.txt', 'page': 1}, page_content='This is a random document')

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../data/algo_book.pdf")

docs = list(loader.lazy_load())

In [2]:
import re
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("../data/algo_book.pdf")
docs = list(loader.lazy_load())

def clean_clrs_page(text: str) -> str: # text is a single page in the book
    text = re.sub(r'(\n[A-Z\-]+\(.*?\)\n)', r'\n<<<ALGO_START>>>\1', text) # pattern example: HEAP-EXTRACT-MAX(A)
    text = re.sub(r'\n\d+\s+Chapter \d+.*?\n', '\n', text) # removes page headers like 123   Chapter 6 Heapsort
    text = re.sub(r'\nProblems for Chapter \d+.*', '', text, flags=re.DOTALL) # removes the exercise questions from the page
    # NEW: remove figure captions — "Figure 6.2  The action of MAX-HEAPIFY..."
    text = re.sub(r'\nFigure \d+\.\d+.*?\n', '\n', text)
    # NEW: collapse 3+ newlines into 2 (PyPDF often over-spaces math blocks)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(
    r'\n(Theorem|Lemma|Definition|Corollary)\s+\d+\.\d+',
    r'\n<<<THEOREM_START>>>\n\1',
    text
    )
    return text

def remove_exercises(docs):
    theory_docs = []
    for doc in docs:
        page_num = doc.metadata.get("page", 0)
        # CLRS 4th ed: theory is roughly pages 28–1246, skip front/back matter
        if page_num < 28 or page_num > 1246:
            continue
        content = doc.page_content
        exercise_split = re.split(r'\n(?=Exercises\s+\d+\.\d+\s*\n)', content) # splits and returns an array where exercise_split[0] is the theory content and exercise_split[1] is the exercises content, handled inline exercise references within theory, ignoring them
        doc.page_content = clean_clrs_page(exercise_split[0]) # clean the theory content and update the document
        if len(doc.page_content.strip()) > 100: # only keep pages with substantial theory content (arbitrary threshold)
            theory_docs.append(doc)
    return theory_docs

clean_docs = remove_exercises(docs)
print(f"Pages after cleaning: {len(clean_docs)}")


Pages after cleaning: 1213


In [3]:
# ── OPEN SOURCE TOKEN LENGTH FUNCTION ─────────────────────────────────────
# QWEN Embedding model — best open source model for RAG embeddings
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B") #downloads and loads the tokenizer associated with this particular model

def token_length(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False)) # the encode function returns the token Ids List like [101, 102, 103, ...] that correspond to the and we take the length of that list to get the token count

# ── CHUNK ──────────────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,        # 512 tokens ≈ full theorem + proof in CLRS
    chunk_overlap=80,      # overlap to preserve reasoning context
    separators=[
        "\n<<<ALGO_START>>>", # custom separator for algorithm starts to keep them intact
        "\n<<<THEOREM_START>>>", # custom separator for theorem/lemma/definition starts to keep them intact
        "\n\n\n",          # section breaks
        "\n\n",            # paragraph breaks
        "\n",              # line breaks (preserves pseudocode structure)
        ". ",              # sentence boundary
        " ",               # word boundary
        "",                # character fallback (last resort)
    ],
    length_function=token_length,
)

chunks = splitter.split_documents(clean_docs)
print(f"Total chunks: {len(chunks)}")

for i, chunk in enumerate(chunks[:5]):
    print(f"\nChunk {i}:")
    print(f"  Tokens : {token_length(chunk.page_content)}")
    print(f"  Page   : {chunk.metadata.get('page')}")
    print(f"  Preview: {chunk.page_content[:80].strip()}...")

Total chunks: 1484

Chunk 0:
  Tokens : 468
  Page   : 28
  Preview: Output: A permutation (reordering) 
  of the input sequence
such that 
 .
Thus,...

Chunk 1:
  Tokens : 402
  Page   : 29
  Preview: Sorting is by no means the only computational problem for which
algorithms have...

Chunk 2:
  Tokens : 448
  Page   : 30
  Preview: (covered in Chapter 31), which are based on numerical algorithms
and number theo...

Chunk 3:
  Tokens : 436
  Page   : 31
  Preview: n! denotes the factorial function. Because the factorial function
grows faster t...

Chunk 4:
  Tokens : 401
  Page   : 32
  Preview: rail network because taking shorter paths results in lower labor
and fuel costs....


### Embedding the chunked tokens to vectors

In [6]:
import os
import torch # used for hardware detection
from langchain_huggingface import HuggingFaceEmbeddings

class EmbeddingManager:
    """Responsible for converting text/chunks into embeddings using Qwen3."""
    
    def __init__(
        self,
        model_name: str = "Qwen/Qwen3-Embedding-0.6B",
        batch_size: int = 32, # defines how many batches to embed at once, larger batch sizes can speed up embedding but require more GPU memory, adjust based on your hardware capabilities
        device: str = None,
    ):
        self.batch_size = batch_size
        if device:
            self.device = device
        elif torch.backends.mps.is_available():
            self.device = "mps"
        elif torch.cuda.is_available():
            self.device = "cuda"
        else:
            self.device = "cpu"
        
        print(f"Loading embedding model on {self.device}...")
        self.model = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={"device": self.device},
            encode_kwargs={
                "normalize_embeddings": True,  # required for Qwen3, divides the vectors by their length, useful for cosine similarity in FAISS
                "batch_size": self.batch_size,
            }
        )

    def embed_chunks(self, chunks: list) -> tuple[list, list]:
        """
        Convert document chunks into embeddings.
        Returns (texts, embeddings) — keeps them paired for FAISS ingestion.
        """
        print(f"Embedding {len(chunks)} chunks on {self.device}...")
        texts = [chunk.page_content for chunk in chunks]
        embeddings = self.model.embed_documents(texts)
        print(f"Done — {len(embeddings)} embeddings of dim {len(embeddings[0])}")
        return chunks, embeddings

    def embed_query(self, query: str) -> list[float]:
        """Convert a single query string into an embedding for retrieval."""
        return self.model.embed_query(query)

In [9]:
import os
from langchain_community.vectorstores import FAISS

class VectorStoreManager:
    """Responsible for storing, persisting, and retrieving from FAISS index."""

    def __init__(self, index_path: str = "../data/clrs_faiss_index"):
        self.index_path = index_path
        self.vectorstore = None

    def build(self, chunks, embeddings):

        print("Building FAISS index...")

        texts = [chunk.page_content for chunk in chunks]
        metadatas = [chunk.metadata for chunk in chunks]

        pairs = list(zip(texts, embeddings))

        self.vectorstore = FAISS.from_embeddings(
            pairs,
            embedding=None,
            metadatas=metadatas
        )

        os.makedirs(os.path.dirname(self.index_path), exist_ok=True)

        self.vectorstore.save_local(self.index_path)

        print(f"Index saved → {self.index_path} ({self.total_vectors} vectors)")

    def load(self, embedding_manager: EmbeddingManager) -> None:
        """Load persisted FAISS index from disk."""
        if not os.path.exists(self.index_path):
            raise FileNotFoundError(
                f"No index at {self.index_path}. Call build() first."
            )
        self.vectorstore = FAISS.load_local( # loads index from disk created from build
            self.index_path,
            embeddings=embedding_manager.model, # re-instantiate the embedding model to ensure compatibility with the loaded index, especially important if using a custom or updated model
            allow_dangerous_deserialization=True
        )
        print(f"Loaded FAISS index — {self.total_vectors} vectors")

    def get_or_build(self, embedding_manager: EmbeddingManager, chunks: list = None) -> None:
        """Load index if it exists, otherwise build from chunks."""
        if os.path.exists(self.index_path):
            self.load(embedding_manager)
        elif chunks is not None:
            chunks, embeddings = embedding_manager.embed_chunks(chunks)
            self.build(chunks, embeddings)
        else:
            raise ValueError("No index found and no chunks provided to build one.")

    def search(self, query_embedding: list[float], k: int = 4) -> list:
        """Retrieve top-k chunks by a pre-computed query embedding."""
        self._guard()
        return self.vectorstore.similarity_search_by_vector(query_embedding, k=k)

    def search_with_score(self, query_embedding: list[float], k: int = 4) -> list:
        """Retrieve top-k chunks with L2 distance scores (lower = more similar)."""
        self._guard()
        return self.vectorstore.similarity_search_by_vector_with_relevance_scores( # returns a list of tuples [(Document, score), ...] where score is the L2 distance between the query embedding and the chunk embedding, lower scores indicate higher similarity
            query_embedding, k=k
        )

    @property
    def total_vectors(self) -> int:
        return self.vectorstore.index.ntotal if self.vectorstore else 0

    def _guard(self) -> None:
        if self.vectorstore is None:
            raise RuntimeError("No vectorstore loaded. Call get_or_build() first.")

In [ ]:
embedder = EmbeddingManager()
store = VectorStoreManager(index_path="../data/clrs_faiss_index")

# Build once, load on every subsequent run
store.get_or_build(embedder, chunks)

# Query — embedding concern stays in EmbeddingManager
query = "how does heapsort maintain the heap property?"
query_embedding = embedder.embed_query(query)

# Retrieval concern stays in VectorStoreManager
results = store.search(query_embedding, k=4)
for i, doc in enumerate(results):
    print(f"\nResult {i+1}")
    print("Page:", doc.metadata.get("page"))
    print("Text:", doc.page_content[:300])

**Retrieval Pipeline**

In [ ]:
import math
import torch
from typing import Literal
from langchain_community.retrievers import BM25Retriever
from sentence_transformers import CrossEncoder


# ── 1. BM25 INDEX ─────────────────────────────────────────────────────────────

class BM25Index:
    """
    Thin wrapper around LangChain's BM25Retriever.
    Built once from the same `chunks` list used for FAISS.
    """

    def __init__(self, chunks: list, k: int = 20):
        self.k = k
        print(f"Building BM25 index over {len(chunks)} chunks...")
        self.retriever = BM25Retriever.from_documents(chunks)
        self.retriever.k = k
        # Keep a text→chunk map for score lookup later
        self._text_to_chunk = {c.page_content: c for c in chunks}
        print("BM25 index ready.")

    def search(self, query: str) -> list[tuple]:
        """
        Returns [(Document, bm25_rank_position), ...].
        BM25Retriever doesn't expose raw scores, so we use rank position (0-indexed).
        RRF only needs rank, not raw score.
        """
        docs = self.retriever.invoke(query)
        return [(doc, rank) for rank, doc in enumerate(docs)]


# ── 2. RERANKER ───────────────────────────────────────────────────────────────

class CrossEncoderReranker:
    """
    Wraps a HuggingFace cross-encoder for precision reranking.
    Takes (query, chunk_text) pairs and returns relevance scores.
      - "BAAI/bge-reranker-base"                  ← strongest open-source
    """

    def __init__(
        self,
        model_name: str = "BAAI/bge-reranker-base",
        device: str = None, 
        batch_size: int = 32, # how many document objects to rerank at once
    ):
        if device:
            self.device = device
        elif torch.cuda.is_available():
            self.device = "cuda"
        elif torch.backends.mps.is_available():
            self.device = "mps"
        else:
            self.device = "cpu"

        self.batch_size = batch_size
        print(f"Loading reranker '{model_name}' on {self.device}...")
        self.model = CrossEncoder(model_name, device=self.device)
        print("Reranker ready.")

    def rerank(self, query: str, candidates: list[dict]) -> list[dict]: # candidates are the retrieved chunks/ document objects
        """
        Scores each candidate against the query.

        Args:
            query      : the user's question
            candidates : list of dicts with at least {"content": str, ...}

        Returns the same list, sorted descending by "rerank_score",
        with "rerank_score" added to each dict.
        """
        if not candidates:
            return []

        pairs = [[query, c["content"]] for c in candidates] # prepare a list of query and document pair

        # Cross-encoder scores in batches (avoids OOM on large candidate sets)
        scores = []
        for i in range(0, len(pairs), self.batch_size):
            batch_scores = self.model.predict(pairs[i : i + self.batch_size]) #predict the relevancy score of the pair
            scores.extend(batch_scores.tolist()) # scores = [9.1, 4.2, 0.8]

        for candidate, score in zip(candidates, scores): # attach score to each document
            candidate["rerank_score"] = round(float(score), 4)

        return sorted(candidates, key=lambda x: x["rerank_score"], reverse=True) # sort documents by score in descending order


# ── 3. HYBRID RAG RETRIEVER ───────────────────────────────────────────────────

class RAGRetriever:
    """
    Full hybrid retrieval pipeline:

        Query
          ├─ BM25 keyword search  ──┐
          └─ Semantic (FAISS)     ──┴─► RRF fusion ──► MMR dedup ──► Rerank ──► top-k

    Args:
        vector_store      : your VectorStoreManager (FAISS)
        embedding_manager : your EmbeddingManager   (Qwen3)
        bm25_index        : your BM25Index
        reranker          : your CrossEncoderReranker
        k                 : final number of chunks to return
        fetch_k           : candidates fetched per leg before fusion (should be >> k)
        mmr_lambda        : 0 = max diversity, 1 = max relevance (MMR dedup step)
        rrf_k             : RRF constant (60 is the standard default)
        score_threshold   : drop final chunks with rerank_score below this (optional)
    """

    def __init__(
        self,
        vector_store: VectorStoreManager, # vector store for semantic search
        embedding_manager: EmbeddingManager, # for converting query to embeddings
        bm25_index: BM25Index, # to implement keyword search
        reranker: CrossEncoderReranker, # to rerank the retriever's results
        k: int = 6, # final number of chunks returned as context
        fetch_k: int = 20, # number of candidates before reranking
        mmr_lambda: float = 0.5,
        rrf_k: int = 60,
        score_threshold: float = None,
    ):
        self.vs = vector_store
        self.em = embedding_manager
        self.bm25 = bm25_index
        self.reranker = reranker
        self.k = k
        self.fetch_k = fetch_k
        self.mmr_lambda = mmr_lambda
        self.rrf_k = rrf_k
        self.score_threshold = score_threshold

    # ── PUBLIC API ─────────────────────────────────────────────────────────

    def retrieve(self, query: str) -> list[dict]:
        """
        Full pipeline. Returns a list of dicts:
            {
                "content"      : str,
                "page"         : int,
                "rrf_score"    : float,   # after fusion
                "rerank_score" : float,   # after cross-encoder
            }
        """
        # Step 1 — gather candidates from both legs
        semantic_hits = self._semantic_search(query)   # [(doc, score), ...]
        bm25_hits     = self.bm25.search(query)        # [(doc, rank),  ...]

        # Step 2 — fuse with RRF
        fused = self._rrf_fusion(semantic_hits, bm25_hits)  # [dict, ...]

        # Step 3 — MMR deduplication over fused candidates
        diverse = self._mmr_filter(query, fused)

        # Step 4 — cross-encoder rerank
        reranked = self.reranker.rerank(query, diverse)

        # Step 5 — threshold + final top-k
        if self.score_threshold is not None:
            reranked = [r for r in reranked if r["rerank_score"] >= self.score_threshold]

        return reranked[: self.k]

    # ── STEP 1: SEARCH LEGS ────────────────────────────────────────────────

    def _semantic_search(self, query: str) -> list[tuple]:
        """Top fetch_k chunks from FAISS with cosine scores."""
        query_emb = self.em.embed_query(query)
        return self.vs.search_with_score(query_emb, k=self.fetch_k)

    # ── STEP 2: RRF FUSION ─────────────────────────────────────────────────

    def _rrf_fusion( # core idea of rrf, a document is highly relevant if it ranks well in both the retreival mechanisms
        self,
        semantic_hits: list[tuple],  # [(Document, cos_score), ...]
        bm25_hits:     list[tuple],  # [(Document, rank_pos),  ...]
    ) -> list[dict]:
        """
        Reciprocal Rank Fusion:
            rrf_score(d) = Σ  1 / (k + rank_in_list)

        Each unique chunk accumulates scores from both lists.
        Using page_content as the dedup key (same as FAISS / BM25 source).
        """
        scores: dict[str, float]  = {}
        docs:   dict[str, object] = {}  # content → Document

        def _add(ranked_list, key_fn=lambda x: x[0].page_content):
            for rank, item in enumerate(ranked_list):
                doc   = item[0]
                key   = key_fn(item)
                delta = 1.0 / (self.rrf_k + rank + 1)  # +1: ranks are 0-indexed
                scores[key] = scores.get(key, 0.0) + delta
                docs[key]   = doc

        _add(semantic_hits)
        _add(bm25_hits)

        # Sort by descending RRF score and package as dicts
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return [
            {
                "content"  : content,
                "page"     : docs[content].metadata.get("page"),
                "rrf_score": round(score, 6),
                "rerank_score": 0.0,  # placeholder, filled by reranker
            }
            for content, score in ranked
        ]

    # ── STEP 3: MMR DEDUP ──────────────────────────────────────────────────

    def _mmr_filter(self, query: str, candidates: list[dict]) -> list[dict]:
      """
      Greedy MMR — numpy version for speed.
      Embeddings are already L2-normalised, so dot product == cosine similarity.
      """
      import numpy as np

      if not candidates:
          return []

      texts     = [query] + [c["content"] for c in candidates]
      all_embs  = np.array(self.em.model.embed_documents(texts))  # (N+1, dim)
      query_emb = all_embs[0]       # (dim,)
      cand_embs = all_embs[1:]      # (N, dim)

      # Relevance of every candidate to the query — shape (N,)
      rel_scores = cand_embs @ query_emb

      selected_idx  = []
      remaining_idx = list(range(len(candidates)))

      for _ in range(min(self.fetch_k, len(candidates))):
          if not selected_idx:
              # First pick: highest relevance
              best_idx = int(np.argmax(rel_scores))
          else:
              selected_embs = cand_embs[selected_idx]          # (S, dim)
              # Redundancy: max similarity to any already-selected chunk
              red_scores = (cand_embs @ selected_embs.T).max(axis=1)  # (N,)
              mmr_scores = (
                  self.mmr_lambda * rel_scores
                  - (1 - self.mmr_lambda) * red_scores
              )
              # Only consider remaining (not yet selected) candidates
              mask           = np.full(len(candidates), -np.inf)
              mask[remaining_idx] = mmr_scores[remaining_idx]
              best_idx       = int(np.argmax(mask))

          selected_idx.append(best_idx)
          remaining_idx.remove(best_idx)

      return [candidates[i] for i in selected_idx]

    # ── UTILS ──────────────────────────────────────────────────────────────

    def __repr__(self) -> str:
        return (
            f"RAGRetriever(k={self.k}, fetch_k={self.fetch_k}, "
            f"mmr_lambda={self.mmr_lambda}, rrf_k={self.rrf_k}, "
            f"vectors={self.vs.total_vectors})"
        )